# Transformation des données

- neo4j pwd : PdEfWwvxMguY_mAK1QcJ8ZUHduJMYbT0DB1fNB_8p0Y

In [1]:
%pip -q install -U ipykernel
%pip -q install pandas

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
path_in   = "/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/base/"
#path_load = "M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/"
path_load = "/Users/asriel/Library/Application Support/Neo4j Desktop/Application/relate-data/dbmss/dbms-e8fc135f-9868-4839-8273-4c0d7fb6f112/"
path_prep = "/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/prep_data/"  
path_tweet = "/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_2/" 
# server.directories.import=import 

In [3]:
import pandas as pd
from datetime import datetime

# Nettoyage des dates 

In [4]:


def parse_date_with_time_range(date_str):
    if pd.isna(date_str):
        return "unknown"
    
    date_str = str(date_str).strip()
    
    # Supprimer la partie horaire
    if '—' in date_str:
        date_part = date_str.split('—')[0].strip()
    else:
        date_part = date_str
    
    # Gérer les plages de dates
    if '–' in date_part:
        start, end = date_part.split('–')
        start = start.strip()
        end = end.strip()
        
        # Si la date de début n'inclut pas le mois ou l'année, les ajouter depuis la date de fin
        if len(start.split()) == 1:
            start = f"{start} {' '.join(end.split()[1:])}"
        elif len(start.split()) == 2:
            start = f"{start} {end.split()[-1]}"
        
        try:
            start_date = datetime.strptime(start, "%d %B %Y").strftime("%Y-%m-%d")
            end_date = datetime.strptime(end, "%d %B %Y").strftime("%Y-%m-%d")
            return f"{start_date} to {end_date}"
        except ValueError:
            return date_str
    
    # Gérer les dates simples
    try:
        return datetime.strptime(date_part, "%d %B %Y").strftime("%Y-%m-%d")
    except ValueError:
        return date_str

In [5]:
from datetime import datetime
import pandas as pd

def standardize_single_date(date_str, year=None):
    if pd.isna(date_str) or date_str == '—':
        return "unknown"
    
    date_str = str(date_str).strip()
    
    try:
        if len(date_str.split()) == 2:  # Si seulement jour et mois
            date_str = f"{date_str} {year}"
        return datetime.strptime(date_str, "%d %B %Y").strftime("%Y-%m-%d")
    except ValueError:
        return "unknown"

def standardize_date_range(date_str, year):
    if pd.isna(date_str) or date_str == '—':
        return "unknown", "unknown"
    
    if '–' in date_str:
        start, end = date_str.split('–')
        start_date = standardize_single_date(start.strip(), year)
        end_date = standardize_single_date(end.strip(), year)
        return start_date, end_date
    return "unknown", "unknown"

def process_olympic_dates(row):
    year = row['year']
    
    # Traiter d'abord competition_date
    comp_start, comp_end = standardize_date_range(row['competition_date'], year)
    
    # Standardiser et compléter start_date
    row['start_date'] = standardize_single_date(row['start_date'], year)
    if row['start_date'] == "unknown" and comp_start != "unknown":
        row['start_date'] = comp_start
        
    # Standardiser et compléter end_date
    row['end_date'] = standardize_single_date(row['end_date'], year)
    if row['end_date'] == "unknown" and comp_end != "unknown":
        row['end_date'] = comp_end
    
    # Mettre à jour competition_date au format standardisé
    if row['start_date'] != "unknown" and row['end_date'] != "unknown":
        row['competition_date'] = f"{row['start_date']} to {row['end_date']}"
    
    return row


In [6]:
data_0 = pd.read_csv(path_in + 'Olympic_Games.csv')
data_0['edition_id'] = 'ed_' + data_0['edition_id'].astype(str)
data_0.to_csv(path_prep + 'Olympic_Games_Standardized.csv', index=False)

In [7]:
data_1 = pd.read_csv(path_in + 'Olympic_Athlete_Event_Results.csv') 
data_1['edition_id'] = 'ed_' + data_1['edition_id'].astype(str)
data_1.to_csv(path_prep + 'Olympic_Athlete_Event_Results_Standardized.csv', index=False)

In [8]:
data_2 = pd.read_csv(path_in + 'Olympic_Games_Medal_Tally.csv') 
data_2['edition_id'] = 'ed_' + data_2['edition_id'].astype(str)
data_2.to_csv(path_prep + 'Olympic_Games_Medal_Tally_Standardized.csv', index=False)

In [9]:
data_3 = pd.read_csv(path_in + 'Olympic_Results.csv')    
data_3['edition_id'] = 'ed_' + data_3['edition_id'].astype(str)
data_3.to_csv(path_prep + 'Olympic_Results_Standardized.csv', index=False)

In [10]:
# Application du traitement de standardisation des dates
data = pd.read_csv(path_prep + 'Olympic_Games_Standardized.csv')

data = data.apply(process_olympic_dates, axis=1)
# Sauvegarde du résultat
data.to_csv(path_prep + 'Olympic_Games_Standardized.csv', index=False)

In [11]:
data = pd.read_csv(path_in + 'Olympic_Athlete_Bio.csv')

for column in ['born']:
    data[column] = data[column].fillna('unknown').apply(parse_date_with_time_range) 
# Sauvegarder dans un nouveau fichier
data.to_csv(path_prep + 'Olympic_Athlete_Bio_Standardized.csv', index=False)

In [12]:
data = pd.read_csv(path_prep + 'Olympic_Results_Standardized.csv')

for column in ['result_date']:
    data[column] = data[column].fillna('unknown').apply(parse_date_with_time_range)   
# Sauvegarder dans un nouveau fichier
data.to_csv(path_prep + 'Olympic_Results_Standardized.csv', index=False)

In [13]:
olympics_country = pd.read_csv(path_in + 'Olympic_Country.csv')
if 'UNK' not in olympics_country['noc'].values:
    # Créer un nouveau DataFrame avec l'enregistrement à ajouter
    new_row = pd.DataFrame({'noc': ['UNK'], 'country': ['UNKNOWN']})
    # Concaténer le nouveau DataFrame avec olympics_country
    olympics_country = pd.concat([olympics_country, new_row], ignore_index=True) 
# Sauvegarder dans un nouveau fichier
olympics_country.to_csv(path_prep + 'Olympic_Country_Standardized.csv', index=False)

In [14]:
import re
# exract a string r'\b(Summer)\b'
# extract year r'\b(\d{4})\b'
# year = extract_pattern("2019 Summer Olympics", r'\b(\d{4})\b')
def extract_pattern(string, pattern):
    match = re.search(pattern, string)
    if match:
        return match.group(1)
    return None

def calculate_age_participation(row):
    edition = extract_pattern(row['edition'],r'\b(\d{4})\b')
    if pd.isna(row['born']) or pd.isna(edition):
#        print(f" date_of_birth: {row['born']} edition : {row['edition_x']}")
        return 'unknown'
    else:
        try:
            birth_year = pd.to_datetime(row['born']).year
            age = int(edition) - int(birth_year)
#            print(f"{row['athlete']} : birth_year {birth_year}  : edition : {edition}  :  age : {age}")
            return age
        except:
            return 'unknown'

LEFT JOIN Olympic_Athlete_event_Results.csv file with Olympic_Athlete_Bio.csv (on
athlete_id) for complete information of the athlete (i.e., height, weight, date of birth
when participating in the event)

In [15]:
from datetime import datetime

# Charger les import CSV
athlete_events = pd.read_csv(path_prep + 'Olympic_Athlete_Event_Results_Standardized.csv')
results = pd.read_csv(path_prep + 'Olympic_Results_Standardized.csv')

# Première jointure pour les informations des athlètes
df = pd.merge(
    athlete_events,
    results,
    on='result_id',
    how='left'
)
#print(df.columns)
df['edition'] = df['edition_x'].fillna(df['edition_y'])
df = df.drop(['edition_x', 'edition_y'], axis=1)

df['sport'] = df['sport_x'].fillna(df['sport_y'])
df = df.drop(['sport_x', 'sport_y'], axis=1)

df['edition_id'] = df['edition_id_x'].fillna(df['edition_id_y'])
df = df.drop(['edition_id_x', 'edition_id_y'], axis=1)

columns=['edition', 'edition_id', 'result_id','athlete','athlete_id',
         'country_noc','sport','event','pos','medal', 'sport_url','result_date', 
         'result_location', 'result_participants','result_format', 'result_detail', 'result_description']

df = df[columns]
df.to_csv(path_prep + 'Olympic_Athlete_Event_Results_Prepared.csv', index=False)
print(df.columns)

Index(['edition', 'edition_id', 'result_id', 'athlete', 'athlete_id',
       'country_noc', 'sport', 'event', 'pos', 'medal', 'sport_url',
       'result_date', 'result_location', 'result_participants',
       'result_format', 'result_detail', 'result_description'],
      dtype='object')


In [16]:
from datetime import datetime

# Charger les import CSV
athlete_events = pd.read_csv(path_prep + 'Olympic_Athlete_Event_Results_Standardized.csv')
athlete_bio = pd.read_csv(path_prep + 'Olympic_Athlete_Bio_Standardized.csv')

# Première jointure pour les informations des athlètes
df = pd.merge(
    athlete_events,
    athlete_bio,
    on='athlete_id',
    how='left'
)
df['country_noc'] = df['country_noc_x'].fillna(df['country_noc_y'])
df = df.drop(['country_noc_x', 'country_noc_y'], axis=1)

#print(df.columns)
# Extraire l'année et la saison de edition_x
df['year'] = df['edition'].str.extract(r'(\d{4})')
df['season'] = df['edition'].apply(lambda x: extract_pattern(x, r'\d{4}\s+(\w+)\s+Olympics'))

# Calcul du BMI
# Convertir les colonnes en numérique
df['weight'] = pd.to_numeric(df['weight'], errors='coerce')
df['height'] = pd.to_numeric(df['height'], errors='coerce')

# Calculer le BMI après conversion
df['bmi'] = df['weight'] / ((df['height']/100) ** 2)

# Identifier les colonnes avec des données manquantes
missing_values  = df.isnull().sum()
missing_columns = missing_values[missing_values > 0]

# Calcul de l'âge
df['age_participation'] = df.apply(calculate_age_participation, axis=1)

columns=['edition', 'edition_id','athlete','athlete_id','sex','born','country_noc',
         'country','age_participation','bmi','height','weight','sport','event',
         'result_id','pos','medal', 'description', 'special_notes']
df = df[columns]

df.to_csv(path_prep + 'Olympic_Athlete_Bio_Prepared.csv', index=False)
print(df.columns)

Index(['edition', 'edition_id', 'athlete', 'athlete_id', 'sex', 'born',
       'country_noc', 'country', 'age_participation', 'bmi', 'height',
       'weight', 'sport', 'event', 'result_id', 'pos', 'medal', 'description',
       'special_notes'],
      dtype='object')


In [17]:

# Nous créeons un index qui soit unique pour chacune des occurences edition, athlete et result_id que nous ajoutons au dataframe des résultats.
olympic_result  = pd.read_csv(path_prep + 'Olympic_Athlete_Event_Results_Prepared.csv')
distinct_result = olympic_result.drop_duplicates(['edition_id','result_id','athlete_id']).reset_index(drop=True)
distinct_event = olympic_result[['edition_id', 'result_id', 'sport', 'event']].drop_duplicates(['result_id']).reset_index(drop=True)
print(len(distinct_event))

7397


In [18]:
# Définit le noeud et les données ATHLETE une entrée par athlete_id avec le premier nom ne tient pas compte des mariages
def process_athlete(distinct_athletes):    
    with \
         open(path_load + "import/" + 'borned_in.csv','w', newline='') as _borned_in_file, \
         open(path_load + "import/" + 'athlete.csv', 'w', newline='') as _athlete_file:  

         _athlete_file.write(f"athlete_id:ID,name,sex,born,bmi,age_participation, :LABEL\n")    
         _borned_in_file.write(f":START_ID,:END_ID, :TYPE\n")
         for _, row in distinct_athletes.iterrows():
            athlete                = row["athlete"].strip().replace(',', '.')
            name                   = f'{athlete}'
            sex                    = str(row['sex']).strip().replace(',', '.')
            born                   = str(row['born']).strip().replace(',', '.')
            bmi                    = str(row['bmi']).strip().replace(',', '.')
            age_participation      = str(row['age_participation']).strip().replace(',', '.')
            athlete_id = row['athlete_id']
            _athlete_file.write(f"{athlete_id}, {name}, {sex},{born},{bmi},{age_participation}, ATHLETE\n")
            _borned_in_file.write(f"{row['athlete_id']}, {row['country_noc'].strip('')}  , BORNED_IN\n")

athlete_results  = pd.read_csv(path_prep + 'Olympic_Athlete_Bio_Prepared.csv')
distinct_athletes = athlete_results.drop_duplicates(subset=['athlete_id'], keep='first').reset_index(drop=True)  
print("Colums athlete : ",athlete_results.columns)
process_athlete(distinct_athletes)

Colums athlete :  Index(['edition', 'edition_id', 'athlete', 'athlete_id', 'sex', 'born',
       'country_noc', 'country', 'age_participation', 'bmi', 'height',
       'weight', 'sport', 'event', 'result_id', 'pos', 'medal', 'description',
       'special_notes'],
      dtype='object')


In [19]:
# Définit le noeud et les données COUNTRY 
def process_country(distinct_country):    
    with \
        open(path_load + "import/"  + 'country.csv', 'w', newline='') as _country_file:

        _country_file.write(f"country_id:ID,name, :LABEL\n")
        for _, value in distinct_country.iterrows():
            country_noc = value['noc']
            country     = value["country"].strip().replace(',', '.')  
            country     = f'{country}'
            _country_file.write(f"{country_noc},{country},COUNTRY\n")

olympics_country = pd.read_csv(path_prep + 'Olympic_Country_Standardized.csv')
distinct_country = olympics_country[['noc','country']].drop_duplicates().reset_index(drop=True)
print("Colums : ",olympics_country.columns)
#process_country(distinct_country)

Colums :  Index(['noc', 'country'], dtype='object')


In [20]:
import pandas as pd
import re

def clean_hashtags(row):
    if isinstance(row, str):  # Vérifier si la valeur est une chaîne de caractères
        return re.sub(r"[\[\]',]", "", row)

# # Extraire les tweets uniques
def process_tweet(distinct_tweet):
    with open(path_load + "import/"  + "tweets.csv", "w") as _tweet_file:
        
        _tweet_file.write(f"tweet_id:ID,hashtags,date,user_name, :LABEL\n")
        for i, value in distinct_tweet.iterrows():
            date       = value["date"]
            user_name  = str(value["user_name"]).strip().replace(',', '.')
            user_name  = user_name.replace("'", ' ') 
            hash_tags  = value["hashtags"]
            hashtags   = clean_hashtags(hash_tags)
            tweet_id   = value["id"]
            _tweet_file.write(f"{tweet_id},{hashtags},{date},{user_name}, TWEETS\n")

olympic_tweet = pd.read_csv(
        path_tweet + 'tokyo_2020_tweets.csv',
        sep=',' , encoding='utf-8',engine='python')
distinct_tweet = olympic_tweet.drop_duplicates(['id']).reset_index(drop=True)
print("Colums tweet : ",olympic_tweet.columns)
process_tweet(distinct_tweet)

Colums tweet :  Index(['id', 'user_name', 'user_location', 'user_description', 'user_created',
       'user_followers', 'user_friends', 'user_favourites', 'user_verified',
       'date', 'text', 'hashtags', 'source', 'retweets', 'favorites',
       'is_retweet'],
      dtype='object')


In [21]:
# Définit le noeud et les données CITY et la relation LOCATED_IN entre CITY et COUNTRY
def process_city(distinct_city):    
    with \
        open(path_load + "import/"  + 'city.csv', 'w', newline='') as _city_file:

        _city_file.write(f'city_id:ID,name, :LABEL\n')
        for _, value in distinct_city.items():
            city        = value.strip().replace(',', '.') 
            city_id     = city_id_map.get(value)
            _city_file.write(f'{city_id}, {city}, CITY\n')

olympics_games = pd.read_csv(path_prep + 'Olympic_Games_Standardized.csv')
distinct_city  = olympics_games['city'].drop_duplicates().reset_index(drop=True)
city_id_map    = {value.strip().replace(',', '.'): f'ci_{i+1}' for i, value in distinct_city.items()}
print("Colums city : ",olympics_games.columns)
#process_city(distinct_city)

Colums city :  Index(['edition', 'edition_id', 'edition_url', 'year', 'city',
       'country_flag_url', 'country_noc', 'start_date', 'end_date',
       'competition_date', 'isHeld'],
      dtype='object')


<p style="text-align: center">
<img src="images/model.png" alt="Olympics Games" width=800 large=650/>
</p>

In [22]:
# Définit le noeud et les données CITY et la relation LOCATED_IN entre CITY et COUNTRY
def process_cityRelShip(distinct_cityCntry):    
    with \
        open(path_load + "import/"  + 'located_in.csv', 'w', newline='') as _located_in_file, \
        open(path_load + "import/"  + 'organized_for.csv','w', newline='') as _organize_for_file:

        _located_in_file.write(f':START_ID,:END_ID, :TYPE\n')
        _organize_for_file.write(f':START_ID,:END_ID, :TYPE\n') 

        for _, row in distinct_cityCntry.iterrows():
            edition_id  = row['edition_id']
            city        = city_edID_map.get(edition_id)
            city_id     = city_id_map.get(city)
            country_noc = row['country_noc']
            
            _located_in_file.write(f'{city_id}, {country_noc}, LOCATED_IN\n')
            _organize_for_file.write(f'{edition_id}, {city_id}, ORGANIZED_BY\n')    

olympics_games     = pd.read_csv(path_prep + 'Olympic_Games_Standardized.csv')

distinct_city      = olympics_games['city'].drop_duplicates().reset_index(drop=True)
city_id_map        = {value: f'ci_{i+1}' for i, value in distinct_city.items()}

distinct_cted      = olympics_games[['city','edition_id']].drop_duplicates().reset_index(drop=True)
city_edID_map      = { value['edition_id'] : value['city'] for _, value in distinct_cted.iterrows()}

distinct_edyr      = olympics_games[['edition_id','year']].drop_duplicates().reset_index(drop=True)
year_edID_map      = {value['edition_id'] : value['year'] for _, value in distinct_edyr.iterrows()}

distinct_cityCntry = olympics_games[['edition_id','country_noc']].drop_duplicates().reset_index(drop=True)
#process_cityRelShip(distinct_cityCntry)


In [23]:
 # Définit le noeud MEDAILLE et les données ainsi que la relation WINS entre ATHLETE et MEDAILLE 
def process_medal(distinct_result):    
    with \
        open(path_load + "import/" + 'medal.csv',      'w', newline='') as _medal_file, \
        open(path_load + "import/" + 'participated_to.csv','w', newline='') as _participated_to_file, \
        open(path_load + "import/" + 'harvest.csv',    'w', newline='') as _harvest_file:

        _medal_file.write(f"medal_id:ID, edition_id, type, :LABEL\n")   
        _harvest_file.write(f":START_ID, :END_ID, :TYPE\n") 
        _participated_to_file.write(f":START_ID,:END_ID, :TYPE\n") 

        for i, row in distinct_result.iterrows():
            edition_id   = row["edition_id"]
            athlete_id   = row["athlete_id"]
            result_id    = row["result_id"]
            restat_id    = f'rst_{athlete_id}{result_id}'
            medal_val    = row["medal"]

            event        = row["event"].strip().replace(',', '.') 
            event        = f'{event}'
            medal_id     = f'md_{i+1}'

            _participated_to_file.write(f"{row['athlete_id']}, {row['edition_id'].strip('')}  , PARTICIPATED_TO\n")
            
            if pd.isna(medal_val) or medal_val.lower() not in ["gold", "silver", "bronze"]:
                i -= 1  
            else :      
                _medal_file.write(f"{medal_id},{edition_id},{medal_val}, MEDAL\n") 
                _harvest_file.write(f"{medal_id},{restat_id}, HARVEST\n")

print("Columns medal : ",distinct_result.columns)
#process_medal(distinct_result)

Columns medal :  Index(['edition', 'edition_id', 'result_id', 'athlete', 'athlete_id',
       'country_noc', 'sport', 'event', 'pos', 'medal', 'sport_url',
       'result_date', 'result_location', 'result_participants',
       'result_format', 'result_detail', 'result_description'],
      dtype='object')


In [24]:

# Définit le noeud SPORT et DISCIPLINE ainsi que la relation PART_OF entre SPORT et DISCIPLINE
def process_sport(distinct_sports):  
    with \
        open(path_load + "import/"  + 'sport.csv', 'w', newline='') as _sport_file:

        _sport_file.write(f'sport_id:ID,name, :LABEL\n') 
        for _, row in distinct_sports.iterrows():
            sport      = row["sport"].strip().replace(',', '.')
            sport_id   = sport_id_map.get(sport)  
            _sport_file.write(f'{sport_id}, {sport}, SPORT\n')

distinct_sports  = olympic_result.drop_duplicates('sport')
sport_id_map     = {row['sport']: f'sp_{i+1}' for i, row in distinct_sports.iterrows()}
print("Colums sport : ",distinct_sports.columns)
#process_sport(distinct_sports)

Colums sport :  Index(['edition', 'edition_id', 'result_id', 'athlete', 'athlete_id',
       'country_noc', 'sport', 'event', 'pos', 'medal', 'sport_url',
       'result_date', 'result_location', 'result_participants',
       'result_format', 'result_detail', 'result_description'],
      dtype='object')


In [25]:

# Définit le noeud SPORT et DISCIPLINE ainsi que la relation PART_OF entre SPORT et DISCIPLINE
def process_discipline(distinct_events):  
    with \
        open(path_load + "import/"  + 'discipline.csv',      'w', newline='') as _discipline_file, \
        open(path_load + "import/"  + 'contains.csv',  'w', newline='') as _contains_file:

        _discipline_file.write(f'event_id:ID,name, :LABEL\n')
        _contains_file.write(f':START_ID,:END_ID, :TYPE\n') 

        for i, row in distinct_events.iterrows():
            edition_id = row['edition_id']
            event      = row["event"].strip().replace(',', '.') 
            event_id   = event_id_map[f'{event}']
            sport      = event_sport_map[f'{event}']
            sport_id   = sport_id_map.get(f'{sport}') 

            _discipline_file.write(f'{event_id}, {event}, DISCIPLINE\n')
            _contains_file.write(f'{sport_id}, {event_id}, CONTAINS\n') 

distinct_events  = olympic_result[['edition_id','event','sport']].drop_duplicates('event')
distinct_sports  = olympic_result.drop_duplicates('sport')

event_id_map     = {row['event'].strip().replace(',', '.'): f'ev_{i+1}' for i, row in distinct_events.iterrows()}
sport_id_map     = {row['sport'].strip().replace(',', '.'): f'sp_{i+1}' for i, row in distinct_sports.iterrows()}
event_sport_map  = {row["event"].strip().replace(',', '.'): row['sport'] for _, row in distinct_events.iterrows()}
print("Colums sport : ",distinct_events.columns)
#process_discipline(distinct_events)

Colums sport :  Index(['edition_id', 'event', 'sport'], dtype='object')


In [26]:
# Définit le noeud et les données EDITION  
def process_edition(distinct_editions):    
    with \
        open(path_load + "import/"  + 'edition.csv','w', newline='') as _edition_file:
      
        _edition_file.write(f"edition_id:ID,year, name, start_date, competition_date, isHeld, :LABEL\n")     
        for _, row in distinct_editions.iterrows():
            edition_id           = row['edition_id']  
            year                 = row['year']                      
            edition              = row["edition"].strip().replace(',', '.') 
            name                 = f'{edition}'
            start_date           = row["start_date"].strip().replace(',', '.')
            competition_date     = row["competition_date"].strip().replace(',', '.')
            isHeld               = row["isHeld"]            


            _edition_file.write(f"{edition_id}, {year}, {name}, {start_date}, {competition_date}, {isHeld}, EDITION\n")

olympics_games     = pd.read_csv(path_prep + 'Olympic_Games_Standardized.csv')
distinct_editions  = olympics_games.drop_duplicates().reset_index(drop=True)
print("Colums Edition : ",olympics_games.columns)
#process_edition(distinct_editions)

Colums Edition :  Index(['edition', 'edition_id', 'edition_url', 'year', 'city',
       'country_flag_url', 'country_noc', 'start_date', 'end_date',
       'competition_date', 'isHeld'],
      dtype='object')


In [27]:
# Définit le noeud RESULT et les données associées
def process_result(complete_result):  
    with \
        open(path_load + "import/"  + 'result.csv',          'w', newline='') as _result_to_file, \
        open(path_load + "import/"  + 'performed_by.csv',    'w', newline='') as _performed_by_file, \
        open(path_load + "import/"  + 'competed_in.csv',     'w', newline='') as _competed_in_file,\
        open(path_load + "import/"  + 'tookplace_for.csv',  'w', newline='') as _tookplace_for_file:
       
        _tookplace_for_file.write(f":START_ID,:END_ID, :TYPE\n")           
        _performed_by_file.write(f":START_ID,:END_ID, :TYPE\n")
        _competed_in_file.write(f":START_ID,:END_ID, :TYPE\n")  

        _result_to_file.write(f"restat_id:ID,result_id,pos,result_date,result_location,result_participants, :LABEL\n")  
        
        for i, row in complete_result.iterrows():
            result_id   = row['result_id']
            athlete_id  = row['athlete_id']
            edition_id  = row['edition_id']

            pos                   = str(row['pos']).strip().replace(',', '.') 
            result_date           = str(row['result_date']).strip().replace(',', '.') 
            result_location       = str(row['result_location']).strip().replace(',', '.') 
            result_participants   = str(row['result_participants']).strip().replace(',', '.') 

            restat_id   = f'rst_{athlete_id}{result_id}'
            event       = row["event"].strip().replace(',', '.') 
            event_id    = event_id_map.get(f'{event}') 
         
            _result_to_file.write(f"{restat_id}, {result_id},{pos},{result_date},{result_location},{result_participants}, RESULT\n")            
            _performed_by_file.write(f"{restat_id},{athlete_id}, PERFORMED_BY\n")
            _competed_in_file.write(f" {restat_id}, {event_id}, COMPETED_IN\n") 
            _tookplace_for_file.write(f" {event_id}, {edition_id},  TOOKPLACE_FOR\n") 

print("Colums Result : ",olympic_result.columns)         
#process_result(distinct_result)      

Colums Result :  Index(['edition', 'edition_id', 'result_id', 'athlete', 'athlete_id',
       'country_noc', 'sport', 'event', 'pos', 'medal', 'sport_url',
       'result_date', 'result_location', 'result_participants',
       'result_format', 'result_detail', 'result_description'],
      dtype='object')


<p style="text-align: center">
<img src="images/model.png" alt="Olympics Games" width=800 large=650/>
</p>

In [28]:
# Definir les relation manquantes 
def process_rel_repr_part(distinct_result):
    with \
        open(path_load + "import/"  + 'performed_during_ed.csv', 'w', newline='') as _performed_during_ed_file, \
        open(path_load + "import/"  + 'represent_to.csv',    'w', newline='') as _represent_to_file:
        
        _represent_to_file.write(f":START_ID,:END_ID,:TYPE\n")
        _performed_during_ed_file.write(f":START_ID,:END_ID,:TYPE\n") 
        for i, row in distinct_result.iterrows():
            result_id   = row['result_id']
            edition_id  = row['edition_id']
            # ATHLETE COUNTRY
            athlete_id  = row['athlete_id']
            country_noc = row['country_noc'] 
            # restat_id
            restat_id   = f'rst_{athlete_id}{result_id}'

            _represent_to_file.write(f" {athlete_id}, {country_noc}, REPRESENTS\n")
            _performed_during_ed_file.write(f" {restat_id}, {edition_id}, PERFORM_DURING_ED\n") 

#process_rel_repr_part(distinct_result)

In [29]:
# Créer les import CSV pour les nœuds et les relations
def generate_header():
        headers = {
                #####################################################################################
                'city_header.csv'                      : 'city_id:ID,name, :LABEL',
                'country_header.csv'                   : 'country_id:ID,name, :LABEL',
                'located_in_header.csv'                : ':START_ID,:END_ID, :TYPE',
                #####################################################################################
                'edition_header.csv'                   : 'edition_id:ID,year, name, start_date, competition_date,isHeld, :LABEL',
                'organized_for_header.csv'             : ':START_ID,:END_ID, :TYPE',
                #####################################################################################
                'sport_header.csv'                     : 'sport_id:ID,name, :LABEL',
                'discipline_header.csv'                : 'DISCIPLINE_id:ID,DISCIPLINE, :LABEL', 
                'contains_header.csv'                  : ':START_ID,:END_ID, :TYPE',   
                #####################################################################################
                'athlete_header.csv'                   : 'restat_id:ID,result_id,pos,result_date,result_location,result_participants,:LABEL',
                'competed_in_header.csv'               : ':START_ID,:END_ID, :TYPE',
                'borned_in_header.csv'                 : ':START_ID,:END_ID, :TYPE',        
                ####################################################################################
                'medal_header.csv'                     : 'medal_id:ID,edition_id, type, :LABEL',
                'harvest_header.csv'                   : ':START_ID,:END_ID, :TYPE',
                #####################################################################################
                'result_header.csv'                    : 'restat_id:ID,result_id, :LABEL',
                'represent_to_header.csv'              : ':START_ID,:END_ID, :TYPE', 
                #####################################################################################
                'tweets_header.csv'                    : 'tweet_id:ID,hashtags,date,user_name, :LABEL',
                'performed_by_header.csv'              : ':START_ID,:END_ID, :TYPE',
                'tookplace_for_header.csv'             : ':START_ID,:END_ID, :TYPE',
                'performed_during_ed_header.csv'       : ':START_ID,:END_ID, :TYPE',
                'participated_to_header.csv'           : ':START_ID,:END_ID, :TYPE',
}        
        
        for filename, content in headers.items():
                with open(path_load + "import/" + filename, 'w') as f:
                     f.write(content)
        print("Tous les import CSV et leurs import d'en-tête ont été créés avec succès.")
generate_header()

Tous les import CSV et leurs import d'en-tête ont été créés avec succès.


In [30]:
process_athlete(distinct_athletes)
process_edition(distinct_editions)
process_country(distinct_country)
process_tweet(distinct_tweet)
process_city(distinct_city)
process_cityRelShip(distinct_cityCntry)
process_medal(distinct_result)
process_sport(distinct_sports)
process_discipline(distinct_events)
process_result(distinct_result) 
process_rel_repr_part(distinct_result)
generate_header()

Tous les import CSV et leurs import d'en-tête ont été créés avec succès.


<p style="text-align: center">
<img src="images/model.png" alt="Olympics Games" width=800 large=650/>
</p>

In [31]:
stop

NameError: name 'stop' is not defined

ln -s /Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_CDTSGBDR/TP_PROJET/Partie_1/import import

In [ ]:

'''
bin/neo4j-admin database import full --delimiter="," --quote="'" 
   --nodes=ATHLETE=import/athlete_header.csv, import/athlete.csv 
   --nodes=MEDAL=import/medal_header.csv, import/medal.csv
   --nodes=EDITION=import/edition_header.csv, import/edition.csv
   --nodes=COUNTRY=import/country_header.csv, import/country.csv 
   --nodes=CITY=import/city_header.csv, import/city.csv
   --nodes=DISCIPLINE=import/discipline_header.csv, import/discipline.csv
   --nodes=RESULT=import/result_header.csv, import/result.csv
   --nodes=SPORT=import/sport_header.csv, import/sport.csv      
   --nodes=TWEET=import/tweets_header.csv, import/tweets.csv      
   --relationships=PERFORMED_DURING_ED=import/performed_during_ed_header.csv, import/performed_during_ed.csv 
   --relationships=BORNED_IN=import/borned_in_header.csv, import/borned_in.csv
   --relationships=REPRESENTS=import/represent_to_header.csv, import/represent_to.csv 
   --relationships=ORGANIZED_FOR=import/organized_for_header.csv, import/organized_for.csv
   --relationships=LOCATED_IN=import/located_in_header.csv, import/located_in.csv 
   --relationships=PERFORMED_BY=import/performed_by_header.csv, import/performed_by.csv 
   --relationships=TOOKPLACE_FOR=import/tookplace_for_header.csv, import/tookplace_for.csv 
   --relationships=CONTAINS=import/contains_header.csv, import/contains.csv 
   --relationships=HARVEST=import/harvest_header.csv, import/harvest.csv   
   --relationships=COMPETED_IN=import/competed_in_header.csv, import/competed_in.csv 
   --relationships=PARTICIPATED_TO=import/participated_to_header.csv, import/participated_to.csv 
  --trim-strings=true --multiline-fields=true --overwrite-destination neo4j  --verbose
'''

In [ ]:
%pip install -q ipython-cypher
%pip install -q py2neo
%pip install -q neo4j

In [2]:
from neo4j import GraphDatabase
from py2neo import Graph

In [35]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))
with driver.session() as session:
    result = session.run("CALL db.schema.visualization()")
        # Accéder aux propriétés via le nœud
    print(result)
driver.close() 

In [36]:
# Connexion à la base de données
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))
def get_labels(tx):
    result = tx.run("CALL apoc.meta.stats() YIELD labels RETURN labels")
    return result.single()["labels"]
with driver.session() as session:
    labels = session.execute_read(get_labels)
    print(labels)
driver.close()

{'SPORT': 112, 'RESULT': 315123, 'TWEETS': 159492, 'COUNTRY': 234, 'MEDAL': 44594, 'DISCIPLINE': 964, 'CITY': 45, 'ATHLETE': 155867, 'EDITION': 64}


In [37]:
# La liste des Athlètes
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))
with driver.session() as session:
    result = session.run("MATCH (a:ATHLETE) RETURN \
                         a.name AS name, \
                         a.sex AS sex, \
                         a.athlete_id AS athlete_id \
                         LIMIT 3")
    for record in result:
        print(f"Athlete ID: {record['athlete_id']}, {record['sex']}, Name: {record['name']}")
    result = session.run("MATCH (a:ATHLETE) RETURN COUNT(a) AS athlete_count")
    count = result.single()["athlete_count"]
    print(f"\nNombre total d'athlètes : {count}")
driver.close()        

Athlete ID: 103854, Male, Name: Rony Bakale
Athlete ID: 114548, Male, Name: Ghyd Olonghot
Athlete ID: 115170, Female, Name: Pamela Mouele-Mboussi

Nombre total d'athlètes : 155867


In [38]:
#Voir toutes les éditions  
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))
with driver.session() as session:
    result = session.run("MATCH (a:EDITION)-[ORGANIZED_BY]->(c:CITY)-[:LOCATED_IN]->(p:COUNTRY) RETURN  \
                        a.edition_id AS edition_id, c.name as ville, p.name as pays, \
                        a.name AS edition \
                        LIMIT 3")  
    for record in result:
        print(f"Edition ID: {record['edition_id']}, {record['edition']} at {record['ville']} in {record['pays']}")
    result = session.run("MATCH (a:EDITION) RETURN COUNT(a) AS edition_count")
    count = result.single()["edition_count"]
    print(f"\nNombre total d'édition : {count}")       
driver.close()

Edition ID: ed_1, 1896 Summer Olympics at Athina in Greece
Edition ID: ed_26, 2004 Summer Olympics at Athina in Greece
Edition ID: ed_4, 1906 Intercalated at Athina in Greece

Nombre total d'édition : 64


In [39]:
#Voir toutes les sports  
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))
with driver.session() as session:
    result = session.run("MATCH (a:SPORT) RETURN  \
                        a.sport_id AS sport_id, \
                        a.name AS sport \
                        LIMIT 3")  
    for record in result:
        print(f"Sport ID: {record['sport_id']}, {record['sport']}")
    result = session.run("MATCH (a:SPORT) RETURN COUNT(a) AS sport_count")
    count = result.single()["sport_count"]
    print(f"\nNombre total de sport : {count}")       
driver.close()

Sport ID: sp_1, Athletics
Sport ID: sp_30, Boxing
Sport ID: sp_32, Diving

Nombre total de sport : 112


In [40]:
# Voir tous les résultats
# 314,726 rows of athlete to result data which includes both team sports and individual sports
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))
with driver.session() as session:
    result = session.run("MATCH  (r:RESULT)-[:COMPETED_IN]->(e:DISCIPLINE)-[:CONTAINS]->(s:SPORT) RETURN \
                          r.result_id AS result_id,r.restat_id AS restat_id, e.name AS DISCIPLINE,s.name AS sport LIMIT 3")  
    for record in result:
        print(f"result ID: {record['result_id']}, {record['restat_id']} at {record['DISCIPLINE']} in {record['sport']},")
    result = session.run("MATCH (a:RESULT) RETURN COUNT(a) AS result_count")
    count = result.single()["result_count"]
    print(f"\nNombre total de resultats : {count} avec une différence {count - 314726} par rapport au nombre attendu")       
driver.close()


Nombre total de resultats : 315123 avec une différence 397 par rapport au nombre attendu


<p style="text-align: center">
<img src="images/model.png" alt="Olympics Games" width=600 large=450/>
</p>

In [41]:
#Voir toutes les City  
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))
with driver.session() as session:
    result = session.run("MATCH (a:CITY) RETURN  \
                        a.city_id AS city_id, \
                        a.name AS city \
                        LIMIT 3")  
    for record in result:
        print(f"city ID: {record['city_id']}, {record['city']}")
    result = session.run("MATCH (a:CITY) RETURN COUNT(a) AS city_count")
    count = result.single()["city_count"]
    print(f"\nNombre total de city : {count}")       
driver.close()

city ID: ci_1, Athina
city ID: ci_2, Paris
city ID: ci_3, St. Louis

Nombre total de city : 45


<p style="text-align: center">
<img src="images/model.png" alt="Olympics Games" width=600 large=450/>
</p>

In [42]:
# expected 235 distinct countries (some existing from the past)
# Voir toutes les éditions  
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))
with driver.session() as session:
    result = session.run("MATCH (a:COUNTRY) RETURN  \
                        a.country_id AS country_id, \
                        a.name AS country \
                        LIMIT 3")  
    for record in result:
        print(f"Sport ID: {record['country_id']}, {record['country']}")
    result = session.run("MATCH (a:COUNTRY) RETURN COUNT(a) AS country_count")
    count = result.single()["country_count"]
    print(f"\nNombre total de country : {count}")       
driver.close()

Sport ID: AFG, Afghanistan
Sport ID: ALB, Albania
Sport ID: ALG, Algeria

Nombre total de country : 234


In [35]:
# CALL db.schema.visualization()

In [43]:
#Voir toutes les Médailles  
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))
with driver.session() as session:
    result = session.run("MATCH (a:MEDAL) RETURN  \
                        a.medal_id AS medal_id, \
                        a.type AS medal \
                        LIMIT 3")  
    for record in result:
        print(f"Sport ID:  {record['medal_id']}, {record['medal']}")
    result = session.run("MATCH (a:MEDAL) RETURN COUNT(a) AS medal_count")
    count = result.single()["medal_count"]
    print(f"\nNombre total de medal : {count}")       
driver.close()

Sport ID:  md_51398, Bronze
Sport ID:  md_51399, Bronze
Sport ID:  md_51400, Bronze

Nombre total de medal : 44594


In [44]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))
with driver.session() as session:
    result = session.run("MATCH (n:DISCIPLINE) RETURN n LIMIT 3")
    for record in result:
        node = record['n']
        # Accéder aux propriétés via le nœud
        print(f"{node.get('event_id')}  {node.get('name')}")
driver.close()      

ev_1  100 metres. Men
ev_2  400 metres. Men
ev_3  800 metres. Men


In [45]:
# expected 7326 unique results (result for a specific DISCIPLINE played at an Olympic game)
# Voir toutes les éditions  
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))
with driver.session() as session:
    result = session.run("MATCH (a:DISCIPLINE) RETURN  \
                        a.event_id AS discipline_id, \
                        a.name AS discipline \
                        LIMIT 3")  
    for record in result:
        print(f"Sport ID:  {record['discipline_id']}, {record['discipline']}")
    result = session.run("MATCH (a:DISCIPLINE) RETURN COUNT(a) AS discipline_count")
    count = result.single()["discipline_count"]
    print(f"\nNombre total de DISCIPLINE : {count} ")       
driver.close()

Sport ID:  ev_1, 100 metres. Men
Sport ID:  ev_2, 400 metres. Men
Sport ID:  ev_3, 800 metres. Men

Nombre total de DISCIPLINE : 964 


<p style="text-align: center">
<img src="images/model.png" alt="Olympics Games" width=600 large=450/>
</p>

In [46]:
# Donner le nombre de relations par type ; 
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))
with driver.session() as session:
    result = session.run("MATCH ()-[r]->() RETURN type(r) as relationType, count(*) as count ORDER BY count DESC")  
    for record in result:
               print(f"Nombre : {record['count']} de noeuds par type : {record['relationType']}")          
driver.close()

Nombre : 315123 de noeuds par type : REPRESENTS
Nombre : 315123 de noeuds par type : PARTICIPATED_TO
Nombre : 315123 de noeuds par type : PERFORM_DURING_ED
Nombre : 315123 de noeuds par type : PERFORMED_BY
Nombre : 315123 de noeuds par type : COMPETED_IN
Nombre : 315123 de noeuds par type : TOOKPLACE_FOR
Nombre : 155867 de noeuds par type : BORNED_IN
Nombre : 44594 de noeuds par type : HARVEST
Nombre : 964 de noeuds par type : CONTAINS
Nombre : 64 de noeuds par type : ORGANIZED_BY
Nombre : 64 de noeuds par type : LOCATED_IN


In [47]:
# Donner le nombre de nœuds par label ;  
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))
with driver.session() as session:
    result = session.run("MATCH (n) WITH labels(n) as labels UNWIND labels as label RETURN \
                         DISTINCT label, count(*) as count \
                         ORDER BY label" )  
    for record in result:
        print(f"Nombre : {record['count']} de noeuds par type : {record['label']}")          
driver.close()

Nombre : 155867 de noeuds par type : ATHLETE
Nombre : 45 de noeuds par type : CITY
Nombre : 234 de noeuds par type : COUNTRY
Nombre : 964 de noeuds par type : DISCIPLINE
Nombre : 64 de noeuds par type : EDITION
Nombre : 44594 de noeuds par type : MEDAL
Nombre : 315123 de noeuds par type : RESULT
Nombre : 112 de noeuds par type : SPORT
Nombre : 159492 de noeuds par type : TWEETS


<p style="text-align: center">
<img src="images/model.png" alt="Olympics Games" width=600 large=450/>
</p>

In [48]:
#  Donner les athlètes (nom, pays représenté) qui ont gagné une médaille à l’épreuve « Decathlon, Men » en 2020
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
     result = session.run("""
            MATCH 
                  (med:MEDAL)-[:HARVEST]->(res:RESULT),
                  (res)-[:COMPETED_IN]->(evt:DISCIPLINE),         
                  (res)-[:PERFORM_DURING_ED]->(edt:EDITION),
                  (res)-[:PERFORMED_BY]->(ath:ATHLETE),
                  (ath)-[:REPRESENTS]->(cou:COUNTRY)               
            WHERE edt.year = "2020" AND
                  evt.name = "Decathlon. Men" AND
                  med.type IN ["Gold", "Silver", "Bronze"]       
            RETURN DISTINCT ath.name as Athlete, med.type as Medal, edt.name as Edition, 
                            edt.year as Year, evt.name as DISCIPLINE, cou.name as Country 
            ORDER BY CASE med.type 
                  WHEN "Gold"   THEN 1 
                  WHEN "Silver" THEN 2 
                  WHEN "Bronze" THEN 3 
                        END                                                  
                        """)
     for record in result:
         print(f" {record['Athlete']} pour le {record['Country']} lors de l'édition {record['Edition']} remporte {record['Medal']} dans la discipline {record['DISCIPLINE']}")
driver.close()

 Damian Warner pour le Canada lors de l'édition 2020 Summer Olympics remporte Gold dans la discipline Decathlon. Men
 Kévin Mayer pour le France lors de l'édition 2020 Summer Olympics remporte Silver dans la discipline Decathlon. Men
 Ashley Moloney pour le Australia lors de l'édition 2020 Summer Olympics remporte Bronze dans la discipline Decathlon. Men


In [49]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("MATCH (n) RETURN DISTINCT labels(n), keys(n)")
    for record in result:
        print(record)          
driver.close()

<Record labels(n)=['ATHLETE'] keys(n)=['name', 'sex', 'born', 'bmi', 'age_participation', 'athlete_id']>
<Record labels(n)=['MEDAL'] keys(n)=['edition_id', 'type', 'medal_id']>
<Record labels(n)=['EDITION'] keys(n)=['name', 'edition_id', 'year', 'start_date', 'competition_date', 'isHeld']>
<Record labels(n)=['RESULT'] keys(n)=['result_id', 'pos', 'result_date', 'result_location', 'result_participants', 'restat_id']>
<Record labels(n)=['CITY'] keys(n)=['name', 'city_id']>
<Record labels(n)=['COUNTRY'] keys(n)=['name', 'country_id']>
<Record labels(n)=['DISCIPLINE'] keys(n)=['name', 'event_id']>
<Record labels(n)=['TWEETS'] keys(n)=['hashtags', 'date', 'user_name', 'tweet_id']>
<Record labels(n)=['SPORT'] keys(n)=['name', 'sport_id']>
<Record labels(n)=['TWEETS'] keys(n)=['hashtags', 'date', 'tweet_id']>


<p style="text-align: center">
<img src="images/model.png" alt="Olympics Games" width=600 large=450/>
</p>

In [50]:
#  Donner le nombre d’athlètes féminines en 2016 ;
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
     result = session.run("""
            MATCH (ath:ATHLETE)-[:PARTICIPATED_TO]->(edt:EDITION)      
            WHERE edt.year = "2016" AND ath.sex = "Female"
            RETURN COUNT(DISTINCT ath) AS result_count
                            """)
     count = result.single()["result_count"]
     print(f"\nle nombre d’athlètes féminines en 2016 : {count} ")                                                                      
driver.close()


le nombre d’athlètes féminines en 2016 : 5137 


<p style="text-align: center">
<img src="images/model.png" alt="Olympics Games" width=600 large=450/>
</p>

In [51]:
#Donner tous les athlètes qui ont participé aux jeux pour un pays dans lequel ils ne sont pas nés ;
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
     result = session.run("""
            MATCH (ath:ATHLETE)-[:PARTICIPATED_TO]->(edt:EDITION),
                  (ath)-[:BORNED_IN]->(brn:COUNTRY),
                  (ath)-[:REPRESENTS]->(rep:COUNTRY)
            WHERE brn <> rep
            RETURN DISTINCT ath.name as Athlete, 
                            brn.name as PaysDeNaissance, 
                            rep.name as PaysRepresente
            ORDER BY ath.name LIMIT 3 
                            """)
     for record in result:
         print(f"L'athlète {record['Athlete']} né(e) au {record['PaysDeNaissance']} a représenté(e) {record['PaysRepresente']} ")                                                                      
driver.close()

L'athlète Aaron Cook né(e) au Republic of Moldova a représenté(e) Great Britain 
L'athlète Abbas Khamis né(e) au Egypt a représenté(e) United Arab Republic 
L'athlète Abbas Qali né(e) au Independent Olympic Athletes a représenté(e) Kuwait 


<p style="text-align: center">
<img src="images/model.png" alt="Olympics Games" width=600 large=450/>
</p>

In [52]:
#Donner les disciplines (et les sports associés) qui ont été proposées sur au moins de 10 éditions.
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
     result = session.run("""          
                MATCH  (spr:SPORT)-[:CONTAINS]->(evt:DISCIPLINE),
                       (evt)-[:TOOKPLACE_FOR]->(edt:EDITION)
                WITH evt.name AS disciplineId, spr.name AS SportName, 
                COUNT(DISTINCT edt) AS EditionCount
                WHERE EditionCount > 10
                RETURN disciplineId, SportName, EditionCount
                ORDER BY EditionCount DESC LIMIT 5;
                        """)
     for record in result:
         print(f"Within {record['SportName']} sport, discipline {record['disciplineId']}, has been performed {record['EditionCount']} times during the Olympics Games")                                                                      
driver.close()

Within Athletics sport, discipline 1.500 metres. Men, has been performed 54 times during the Olympics Games
Within Athletics sport, discipline 5.000 metres. Men, has been performed 50 times during the Olympics Games
Within Athletics sport, discipline 10.000 metres. Men, has been performed 50 times during the Olympics Games
Within Archery sport, discipline Individual. Men, has been performed 48 times during the Olympics Games
Within Tennis sport, discipline Singles. Men, has been performed 43 times during the Olympics Games


In [3]:
# Donner les tweets de l’édition 2020 qui concernent le nageur Michael Phelps (hashtag michaelphelps)
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
     result = session.run("""          
                MATCH (tw:TWEETS)
                WHERE tw.hashtags CONTAINS "michaelphelps"
                RETURN tw.tweet_id AS id, tw.hashtags AS hashtags
                        """)
     for record in result:
         print(f"id {record['id']} hashtags {record['hashtags']} ")                                                                      
driver.close()
"""
Colums tweet :  Index(['id', 'user_name', 'user_location', 'user_description', 'user_created',
       'user_followers', 'user_friends', 'user_favourites', 'user_verified',
       'date', 'text', 'hashtags', 'source', 'retweets', 'favorites',
       'is_retweet'],
      dtype='object')
      """

id 1419150908793884673 hashtags Tokyo2020 Olympics UssainBolt michaelphelps 
id 1419192997841969152 hashtags Tokyo2020 Olympics michaelphelps OlympicGames 
id 1419838116228173824 hashtags michaelphelps Tokyo2020 
id 1419878616511430656 hashtags michaelphelps Tokyo2020 


"\nColums tweet :  Index(['id', 'user_name', 'user_location', 'user_description', 'user_created',\n       'user_followers', 'user_friends', 'user_favourites', 'user_verified',\n       'date', 'text', 'hashtags', 'source', 'retweets', 'favorites',\n       'is_retweet'],\n      dtype='object')\n      "

In [4]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("""                    
MATCH (ath:ATHLETE)-[:REPRESENTS]->(cou:COUNTRY),
      (ath)-[:PARTICIPATED_TO]->(edition:EDITION)
WITH edition.year AS Year, cou.name AS Country, COUNT(ath) AS NumberOfAthletes
ORDER BY Year, NumberOfAthletes DESC
WITH Year, collect({Country: Country, Athletes: NumberOfAthletes}) AS CountryStats
RETURN Year, CountryStats[0..5] AS TopCountries LIMIT 10
                         """)
    for record in result:
        print(f"{record['Year']}  {record['TopCountries']}")          
driver.close() 

1896  [{'Country': 'Germany', 'Athletes': 1263}, {'Country': 'Greece', 'Athletes': 618}, {'Country': 'Hungary', 'Athletes': 253}, {'Country': 'Denmark', 'Athletes': 251}, {'Country': 'France', 'Athletes': 246}]
1900  [{'Country': 'France', 'Athletes': 8836}, {'Country': 'United States', 'Athletes': 1309}, {'Country': 'Great Britain', 'Athletes': 826}, {'Country': 'Belgium', 'Athletes': 638}, {'Country': 'Netherlands', 'Athletes': 453}]
1904  [{'Country': 'United States', 'Athletes': 11637}, {'Country': 'Germany', 'Athletes': 312}, {'Country': 'Canada', 'Athletes': 194}, {'Country': 'Hungary', 'Athletes': 147}, {'Country': 'Great Britain', 'Athletes': 83}]
1906  [{'Country': 'Crete', 'Athletes': 107}]
1908  [{'Country': 'Great Britain', 'Athletes': 2535}, {'Country': 'United States', 'Athletes': 1782}, {'Country': 'Sweden', 'Athletes': 1623}, {'Country': 'France', 'Athletes': 1616}, {'Country': 'Hungary', 'Athletes': 996}]
1912  [{'Country': 'Sweden', 'Athletes': 3600}, {'Country': 'Uni

<p style="text-align: center">
<img src="images/model.png" alt="Olympics Games" width=600 large=450/>
</p>

In [5]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("MATCH (n) RETURN DISTINCT labels(n), keys(n)")
    for record in result:
        print(record)          
driver.close()

<Record labels(n)=['ATHLETE'] keys(n)=['name', 'sex', 'born', 'bmi', 'age_participation', 'athlete_id']>
<Record labels(n)=['MEDAL'] keys(n)=['edition_id', 'type', 'medal_id']>
<Record labels(n)=['EDITION'] keys(n)=['name', 'edition_id', 'year', 'start_date', 'competition_date', 'isHeld']>
<Record labels(n)=['RESULT'] keys(n)=['result_id', 'pos', 'result_date', 'result_location', 'result_participants', 'restat_id']>
<Record labels(n)=['CITY'] keys(n)=['name', 'city_id']>
<Record labels(n)=['COUNTRY'] keys(n)=['name', 'country_id']>
<Record labels(n)=['DISCIPLINE'] keys(n)=['name', 'event_id']>
<Record labels(n)=['TWEETS'] keys(n)=['hashtags', 'date', 'user_name', 'tweet_id']>
<Record labels(n)=['SPORT'] keys(n)=['name', 'sport_id']>
<Record labels(n)=['TWEETS'] keys(n)=['hashtags', 'date', 'tweet_id']>


In [6]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("""                    
        MATCH (med:MEDAL)-[:HARVEST]->(res:RESULT),
            (res:RESULT)-[:PERFORMED_BY]->(ath:ATHLETE),                         
            (res)-[:PERFORM_DURING_ED]->(edt:EDITION),                              
            (ath)-[:REPRESENTS]->(cou:COUNTRY)
        WHERE cou.name = "United States" AND edt.year = "2020" AND med.type = "Gold"
        RETURN 
            cou.name AS Country,
            COUNT(DISTINCT med) AS TotalMedals
                         """)
    for record in result:
        print(f"{record['Country']}  {record['TotalMedals']}")          
driver.close() 

United States  113


In [7]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("""                    
            MATCH (med:MEDAL)-[:HARVEST]->(res:RESULT)-[:PERFORMED_BY]->(ath:ATHLETE)-[:REPRESENTS]->(cou:COUNTRY)
            MATCH (res)-[:PERFORM_DURING_ED]->(edition:EDITION)
            RETURN cou.name AS Country, edition.year AS Year,
            COUNT (DISTINCT med.type="Gold") AS Gold,
            COUNT (DISTINCT med.type="Silver") AS Silver,
            COUNT (DISTINCT med.type="Bronze") AS Bronze 
            ORDER BY Year ASC LIMIT 10               
                         """)
    for record in result:
        print(f" {record['Year']}  {record['Country']}  {record['Gold']} {record['Silver']} {record['Bronze']}")          
driver.close() 

 1896  Denmark  2 2 2
 1896  Greece  2 2 2
 1896  Australia  2 1 2
 1896  Great Britain  2 2 2
 1896  Austria  2 2 2
 1896  Switzerland  2 2 1
 1896  Hungary  2 2 2
 1896  Germany  2 2 2
 1896  France  2 2 2
 1896  United States  2 2 2


<p style="text-align: center">
<img src="images/model.png" alt="Olympics Games" width=600 large=450/>
</p>

In [8]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("""                    
            MATCH (ath:ATHLETE)-[:PARTICIPATED_TO]->(edt:EDITION),
                  (ath)-[:REPRESENTS]->(cou:COUNTRY)
            WHERE edt.year = "2020" AND cou.country_id = "USA"
            RETURN DISTINCT ath.name AS Athlete, cou.name AS Country LIMIT 3
                         """)
    for record in result:
        print(f"{record['Country']}  {record['Athlete']}")          
driver.close() 

United States  Amro El-Geziry
United States  Sally Kipyego
United States  Phillip Dutton


<p style="text-align: center">
<img src="images/model.png" alt="Olympics Games" width=600 large=450/>
</p>

In [9]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("""                    
            MATCH (evt:DISCIPLINE)-[:TOOKPLACE_FOR]->(edt:EDITION),
                  (edt)-[:ORGANIZED_BY]->(cit:CITY)
            WHERE edt.year = "2020"
            RETURN DISTINCT evt.name AS DISCIPLINE, cit.name AS City LIMIT 3
                         """)
    for record in result:
        print(f"{record['DISCIPLINE']}  {record['City']}")          
driver.close() 

100 metres. Men  Tokyo
400 metres. Men  Tokyo
800 metres. Men  Tokyo


In [10]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("""                    
        MATCH (tw:TWEETS)
        WHERE tw.hashtags CONTAINS "Tokyo2020"
        RETURN tw.tweet_id AS Tweet, tw.hashtags AS Hashtags LIMIT 5
                         """)
    for record in result:
        print(f"{record['Tweet']}  {record['Hashtags']}")          
driver.close() 

1418882123050754057  SutirthaMukharjee Olympics Tokyo2020 OlympicGames
1418882122992046081  Olympics Tokyo2020
1418882119632359429  Tokyo2020 Olympics RichardCarapaz
1418882115177943042  Tokyo2020
1418882112170647559  Tokyo2020 HUN


In [11]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("""                    
        MATCH (sp:SPORT)-[:CONTAINS]->(evt:DISCIPLINE),
              (evt)-[:TOOKPLACE_FOR]->(ed:EDITION)
        WHERE ed.year = "2020"
        RETURN sp.name AS Sport, COUNT(evt) AS NumberOfDISCIPLINEs
        ORDER BY NumberOfDISCIPLINEs DESC LIMIT 5
                         """)
    for record in result:
        print(f"{record['Sport']}  {record['NumberOfDISCIPLINEs']}")          
driver.close() 

Athletics  2285
Swimming  1622
Artistic Gymnastics  929
Archery  810
Football  608


<p style="text-align: center">
<img src="images/model.png" alt="Olympics Games" width=600 large=450/>
</p>

In [12]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("""                    
        MATCH (med:MEDAL)-[:HARVEST]->(res:RESULT), 
              (res)-[:PERFORMED_BY]->(ath:ATHLETE),
              (ath)-[:REPRESENTS]->(cou:COUNTRY),
              (ath)-[:PARTICIPATED_TO]->(edt:EDITION)
        WHERE edt.year = "2020"
        RETURN cou.name AS Country, 
              med.type AS MedalType, 
        COUNT(DISTINCT med) AS TotalMedals
        ORDER BY TotalMedals DESC LIMIT 5
                         """)
    for record in result:
        print(f"{record['Country']}  {record['MedalType']} {record['TotalMedals']}")          
driver.close() 

United States  Gold 207
United States  Silver 148
United States  Bronze 111
Russian Olympic Committee  Silver 102
Australia  Bronze 94


In [13]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("""                    
                MATCH  (med:MEDAL)-[:HARVEST]->(res:RESULT),
                       (res)-[:PERFORMED_BY]->(ath:ATHLETE),                  
                       (ath)-[:REPRESENTS]->(cou:COUNTRY {country_id: "USA"})
                RETURN DISTINCT ath.name AS AthleteName, med.type AS MedalType
                ORDER BY AthleteName ASC LIMIT 5
                         """)
    for record in result:
        print(f"{record['AthleteName']}  {record['MedalType']} ")          
driver.close() 

A'ja Wilson  Gold 
A. C. Gilbert  Gold 
A. D. Franch  Bronze 
A. J. Hinch  Bronze 
A. J. Mleczko  Gold 


In [14]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run(""" 
            MATCH (ath:ATHLETE)-[:REPRESENTS]->(cou:COUNTRY {country_id: "USA"}),
                  (ath)-[:PARTICIPATED_TO]->(ed:EDITION {year: "2020"})
            RETURN COUNT(DISTINCT ath) AS TotalUSAthlete;
                          """)
    for record in result:
        print(f"Le nombre d'athlètes est de : {record['TotalUSAthlete']} ")          
driver.close() 

Le nombre d'athlètes est de : 628 


<p style="text-align: center">
<img src="images/model.png" alt="Olympics Games" width=600 large=450/>
</p>

## Premier graphique

In [15]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("""   
            MATCH (med:MEDAL)-[:HARVEST]->(res:RESULT),
                  (res)-[:PERFORMED_BY]->(ath:ATHLETE),
                  (res)-[:PERFORM_DURING_ED]->(edt:EDITION),       
                  (ath)-[:REPRESENTS]->(cou:COUNTRY)
            WHERE  cou.country_id = "USA" 
            RETURN cou.name AS Country, 
                  COUNT(DISTINCT med.type = "Gold" ) AS GoldMedals,
                  COUNT(DISTINCT med.type = "Silver") AS SilverMedals, 
                  COUNT(DISTINCT med.type = "Bronze") AS BronzeMedals,              
                  COUNT(DISTINCT med) AS TotalMedals
            ORDER BY TotalMedals DESC
            LIMIT 5;
                         """)
    for record in result:
        print(f"{record['Country']} gold : {record['GoldMedals']} silver : {record['SilverMedals']} bronze : {record['BronzeMedals']} total :  {record['TotalMedals']}")          
driver.close() 

United States gold : 2 silver : 2 bronze : 2 total :  6314


In [16]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("""   
            MATCH (med:MEDAL)-[:HARVEST]->(res:RESULT),
                (res)-[:PERFORMED_BY]->(ath:ATHLETE),
                (ath)-[:REPRESENTS]->(cou:COUNTRY),
                (res)-[:PERFORM_DURING_ED]->(edition:EDITION)
            WHERE cou.country_id = "USA"
            RETURN edition.year AS Year,
                COUNT( med.type = "Gold") AS GoldMedals,
                COUNT( med.type = "Silver") AS SilverMedals,
                COUNT( med.type = "Bronze") AS BronzeMedals
            ORDER BY Year ASC LIMIT 10
                         """)
    for record in result:
        print(f" year  {record['Year']} gold : {record['GoldMedals']} Silver : {record['SilverMedals']} Bronze : {record['BronzeMedals']} ")          
driver.close() 

 year  1896 gold : 79 Silver : 79 Bronze : 79 
 year  1900 gold : 282 Silver : 282 Bronze : 282 
 year  1904 gold : 1699 Silver : 1699 Bronze : 1699 
 year  1908 gold : 397 Silver : 397 Bronze : 397 
 year  1912 gold : 582 Silver : 582 Bronze : 582 
 year  1920 gold : 822 Silver : 822 Bronze : 822 
 year  1924 gold : 523 Silver : 523 Bronze : 523 
 year  1928 gold : 267 Silver : 267 Bronze : 267 
 year  1932 gold : 685 Silver : 685 Bronze : 685 
 year  1936 gold : 232 Silver : 232 Bronze : 232 


In [17]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run(""" 
            MATCH (med:MEDAL)-[:HARVEST]->(res:RESULT),
                  (res)-[:PERFORMED_BY]->(ath:ATHLETE),
                  (res)-[:PERFORM_DURING_ED]->(edt:EDITION),            
                  (ath)-[:REPRESENTS]->(cou:COUNTRY)
            WHERE  cou.country_id = "USA" AND edt.year = "2020"  
            RETURN cou.name AS Country,  
                   med.type AS MedalType,
                   edt.name AS Edition,             
            COUNT(DISTINCT med) AS TotalMedals
            ORDER BY TotalMedals DESC LIMIT 5;
                         """)
    for record in result:
        print(f"{record['Country']} {record['Edition']} harvest {record['TotalMedals']} {record['MedalType']} medals")          
driver.close() 

United States 2020 Summer Olympics harvest 113 Gold medals
United States 2020 Summer Olympics harvest 109 Silver medals
United States 2020 Summer Olympics harvest 75 Bronze medals


<p style="text-align: center">
<img src="images/model_1.png" alt="Olympics Games" width=800 large=650/>
</p>

In [18]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("""                    
            MATCH (med:MEDAL)-[:HARVEST]->(res:RESULT),
                  (res)-[:PERFORMED_BY]->(ath:ATHLETE),
                  (res)-[:PERFORM_DURING_ED]->(edt:EDITION),  
                  (res)-[:COMPETED_IN]->(dis:DISCIPLINE),              
                  (ath)-[:REPRESENTS]->(cou:COUNTRY)
            WHERE cou.country_id = "USA" AND 
                  edt.year = "2020"  AND
                  med.type = "Gold" AND
                  dis.name = "4 × 400 metres Relay. Men"
            RETURN ath.name AS Athlete, 
                  med.type AS MedalType,
                  edt.name AS Edition,
                  dis.name AS Discipline,                 
            COUNT(DISTINCT med) AS GoldMedals
            ORDER BY GoldMedals DESC
                         """)
    for record in result:
        print(f"{record['Athlete']} {record['Edition']} {record['Discipline']} {record['GoldMedals']}  {record['MedalType']}")          
driver.close() 

Michael Cherry 2020 Summer Olympics 4 × 400 metres Relay. Men 1  Gold
Michael Norman 2020 Summer Olympics 4 × 400 metres Relay. Men 1  Gold
Bryce Deadmon 2020 Summer Olympics 4 × 400 metres Relay. Men 1  Gold
Rai Benjamin 2020 Summer Olympics 4 × 400 metres Relay. Men 1  Gold
Trevor Stewart 2020 Summer Olympics 4 × 400 metres Relay. Men 1  Gold
Randolph Ross 2020 Summer Olympics 4 × 400 metres Relay. Men 1  Gold
Vernon Norwood 2020 Summer Olympics 4 × 400 metres Relay. Men 1  Gold


In [19]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("""                    
            MATCH (ath:ATHLETE)-[:REPRESENTS]->(cou:COUNTRY)
            RETURN cou.name AS Country, COUNT(ath) AS NumberOfAthletes
            ORDER BY NumberOfAthletes DESC LIMIT 5
                         """)
    for record in result:
        print(f" {record['Country']}  {record['NumberOfAthletes']}")          
driver.close() 

 United States  23319
 France  16005
 Great Britain  13621
 Italy  12376
 Canada  11688


<p style="text-align: center">
<img src="images/model.png" alt="Olympics Games" width=600 large=450/>
</p>

In [20]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("""                    
            MATCH (sp:SPORT)-[:CONTAINS]->(evt:DISCIPLINE),
                (evt)-[:TOOKPLACE_FOR]->(ed:EDITION)
            WHERE ed.year = "2020"
            RETURN DISTINCT evt.name AS DISCIPLINE, sp.name AS Sport LIMIT 5
                         """)
    for record in result:
        print(f"{record['DISCIPLINE']}  {record['Sport']}")          
driver.close() 

100 metres. Men  Athletics
400 metres. Men  Athletics
800 metres. Men  Athletics
1.500 metres. Men  Athletics
Marathon. Men  Athletics


In [21]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("""                    
            MATCH (tw:TWEETS)
            WHERE tw.hashtags CONTAINS "Tokyo2020"
            RETURN COUNT(tw) AS TotalTweets, COLLECT(DISTINCT tw.hashtags) AS HashtagsUsed
                         """)
    for record in result:
        print(f"{record['TotalTweets']}  {record['HashtagsUsed']}")          
driver.close() 

97832  ['SutirthaMukharjee Olympics Tokyo2020 OlympicGames', 'Olympics Tokyo2020', 'Tokyo2020 Olympics RichardCarapaz', 'Tokyo2020', 'Tokyo2020 HUN', 'Tokyo2020 Cheer4India TokyoOlympics', 'Tokyo2020 TokyoOlympics TeamGB', 'AZE Tokyo2020', 'SWEvAUS Tokyo2020', 'TeamGB Tokyo2020', 'MirabaiChanu india Tokyo2020 Olympics2021', 'OlympicGames Tokyo2020', 'Swimming Tokyo2020 TokyoOlympics Tokyo2021 Olympics OlympicGames', 'TeamGB Tokyo2020 JPNGBR', 'Swimming Tokyo2020 Olympics', 'Tokyo2020 MirabaiChanu SalmanKhan', 'Tokyo2020 TokyoTogether', 'swimming Tokyo2020', 'swimming Olympics Tokyo2020 AUS', 'Gymnastics Tokyo2020', 'BBCOlympics Olympics Tokyo2020', 'Tokyo2020 OlympicGames', 'MirabaiChanu Tokyo2020 Weightlifting', 'Tokyo2020 Olympics Olympics2021', 'Tokyo2020 IndiaAtOlympics', 'NZRepresent Tokyo2020', 'EXCLUSIVELY ITLivestream IndiaTodayatOlympics Tokyo2020', 'Tokyo2020 Cheer4India', 'Tokyo2020 MirabaiChanu', 'BBCOlympics Tokyo2020 Tokyo2020Olympics', 'RingKeBaazigar boxing Tokyo2020 Ch

In [22]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("""                    
            MATCH (ath:ATHLETE)-[:REPRESENTS]->(cou:COUNTRY),
                  (ath)-[:PARTICIPATED_TO]->(edition:EDITION)
            WITH edition.year AS Year, cou.name AS Country, COUNT(ath) AS NumberOfAthletes
            ORDER BY Year, NumberOfAthletes DESC
            WITH Year, collect({Country: Country, Athletes: NumberOfAthletes}) AS CountryStats
            RETURN Year, CountryStats[0..5] AS TopCountries LIMIT 10
                         """)
    for record in result:
        print(f"{record['Year']}  {record['TopCountries']}")          
driver.close() 

1896  [{'Country': 'Germany', 'Athletes': 1263}, {'Country': 'Greece', 'Athletes': 618}, {'Country': 'Hungary', 'Athletes': 253}, {'Country': 'Denmark', 'Athletes': 251}, {'Country': 'France', 'Athletes': 246}]
1900  [{'Country': 'France', 'Athletes': 8836}, {'Country': 'United States', 'Athletes': 1309}, {'Country': 'Great Britain', 'Athletes': 826}, {'Country': 'Belgium', 'Athletes': 638}, {'Country': 'Netherlands', 'Athletes': 453}]
1904  [{'Country': 'United States', 'Athletes': 11637}, {'Country': 'Germany', 'Athletes': 312}, {'Country': 'Canada', 'Athletes': 194}, {'Country': 'Hungary', 'Athletes': 147}, {'Country': 'Great Britain', 'Athletes': 83}]
1906  [{'Country': 'Crete', 'Athletes': 107}]
1908  [{'Country': 'Great Britain', 'Athletes': 2535}, {'Country': 'United States', 'Athletes': 1782}, {'Country': 'Sweden', 'Athletes': 1623}, {'Country': 'France', 'Athletes': 1616}, {'Country': 'Hungary', 'Athletes': 996}]
1912  [{'Country': 'Sweden', 'Athletes': 3600}, {'Country': 'Uni

In [23]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("""
            MATCH (res:RESULT)<-[:HARVEST]-(med:MEDAL),
                  (res)-[:COMPETED_IN]->(dis:DISCIPLINE),
                  (res)-[:PERFORM_DURING_ED]->(edt:EDITION)
            WITH edt.year AS EditionYear, med.type AS MedalType,
                  COUNT(med) AS TotalMedals
            RETURN DISTINCT EditionYear, MedalType, TotalMedals
            ORDER BY EditionYear ASC, MedalType DESC LIMIT 5
                         """)
    for record in result:
        print(f"  {record['EditionYear']}  {record['MedalType']}  {record['TotalMedals']} ")          
driver.close() 

  1896  Silver  46 
  1896  Gold  62 
  1896  Bronze  39 
  1900  Silver  227 
  1900  Gold  212 


<p style="text-align: center">
<img src="images/model.png" alt="Olympics Games" width=800 large=650/>
</p>

In [24]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("""
            MATCH (spo:SPORT)-[:CONTAINS]->(dis:DISCIPLINE),
                   (dis)-[:TOOKPLACE_FOR]->(edt:EDITION)
            RETURN spo.name AS Sport, 
            COUNT(DISTINCT dis) AS DisciplineCount, 
            COUNT(DISTINCT edt) AS EditionCount LIMIT 5
                         """)
    for record in result:
        print(f"  {record['Sport']} {record['DisciplineCount']} {record['EditionCount']} ")          
driver.close() 

  Athletics 154 54 
  Boxing 8 26 
  Diving 10 27 
  Rugby 1 4 
  Shooting 78 28 


In [25]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("""
            MATCH (dis:DISCIPLINE)-[r:TOOKPLACE_FOR]->(edt:EDITION)
            WITH dis.name AS Displine, 
                 edt.edition AS Edition, 
            COUNT(r) AS RelationCount
            WHERE RelationCount > 1
            RETURN Displine, Edition, RelationCount
            ORDER BY RelationCount DESC LIMIT 10
                         """)
    for record in result:
        print(f"  {record['Displine']} {record['Edition']} {record['RelationCount']} ")          
driver.close() 

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: edition)} {position: line: 4, column: 22, offset: 130} for query: '\n            MATCH (dis:DISCIPLINE)-[r:TOOKPLACE_FOR]->(edt:EDITION)\n            WITH dis.name AS Displine, \n                 edt.edition AS Edition, \n            COUNT(r) AS RelationCount\n            WHERE RelationCount > 1\n            RETURN Displine, Edition, RelationCount\n            ORDER BY RelationCount DESC LIMIT 10\n                         '


  Football. Men None 7835 
  Ice Hockey. Men None 5544 
  Hockey. Men None 4574 
  Individual. Men None 4039 
  Water Polo. Men None 3755 
  Basketball. Men None 3577 
  Singles. Men None 3201 
  Road Race. Individual. Men None 3095 
  4 × 100 metres Relay. Men None 2835 
  Singles. Women None 2720 


In [26]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("""
            MATCH (res:RESULT)-[:COMPETED_IN]->(dis:DISCIPLINE),
                  (dis)-[:TOOKPLACE_FOR]->(ed:EDITION)
            WITH  res.result_id AS ResultID, 
                  dis.name AS Result, 
                  COUNT(DISTINCT ed) AS EditionCount
            WHERE EditionCount > 1
            RETURN DISTINCT Result, EditionCount
            ORDER BY EditionCount DESC LIMIT 10
                         """)
    for record in result:
        print(f"  {record['Result']} {record['EditionCount']} ")          
driver.close() 

  1.500 metres. Men 54 
  10.000 metres. Men 50 
  5.000 metres. Men 50 
  Individual. Men 48 
  Singles. Men 43 
  Singles. Women 41 
  Team. Men 39 
  Sprint. Men 33 
  1.500 metres. Women 31 
  Shot Put. Men 30 


In [27]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("""
            MATCH (edition:EDITION)-[:ORGANIZED_BY]->(city:CITY)
            MATCH (city)-[:LOCATED_IN]->(country:COUNTRY)
                WITH city, country
                    CALL {
                        WITH country
                        RETURN country.country_id AS code, count(*) as value
                            UNION
                        WITH city
                        RETURN city.name AS code, count(*) as value
                        }
                WITH code, sum(value) AS totalCount
                RETURN code,totalCount LIMIT 10
                    """)
    for record in result:
        print(f"  {record['code']} {record['totalCount']} ")          
driver.close() 

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: CALL subquery without a variable scope clause is now deprecated. Use CALL (country, city) { ... }} {position: line: 5, column: 21, offset: 179} for query: '\n            MATCH (edition:EDITION)-[:ORGANIZED_BY]->(city:CITY)\n            MATCH (city)-[:LOCATED_IN]->(country:COUNTRY)\n                WITH city, country\n                    CALL {\n                        WITH country\n                        RETURN country.country_id AS code, count(*) as value\n                            UNION\n                        WITH city\n                        RETURN city.name AS code, count(*) as value\n                        }\n                WITH code, sum(value) AS totalCount\n                RETURN code,totalCount LIMIT 10\n                    '


  GRE 9 
  FRA 12 
  Athina 9 
  Paris 9 
  USA 17 
  GBR 16 
  St. Louis 1 
  London 16 
  SWE 4 
  GER 8 


<p style="text-align: center">
<img src="images/model.png" alt="Olympics Games" width=800 large=650/>
</p>

In [28]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("""
            MATCH (res:RESULT)-[:COMPETED_IN]->(dis:DISCIPLINE)
            MATCH (res:RESULT)-[:PERFORM_DURING_ED]->(edi:EDITION)
            MATCH (res:RESULT)-[perf:PERFORMED_BY]->(ath:ATHLETE)
            WHERE dis.name = "1.500 metres. Men" AND edi.year = "2018"
            RETURN DISTINCT ath.name, dis.name, perf LIMIT 3
                """)
    for record in result:
        print(f"  {record['ath.name']}  ")          
driver.close() 

  Choe Un-Song  
  Maksim Siarheyu  
  William Tai  


In [ ]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("""
MATCH (res:RESULT)-[:COMPETED_IN]->(dis:DISCIPLINE)
MATCH (res)-[:PERFORM_DURING_ED]->(edi:EDITION)
MATCH (res)-[:PERFORMED_BY]->(ath:ATHLETE)
WHERE dis.name = "1.500 metres. Men" AND edi.year = "2018"
WITH MAX(res.result_date) AS LatestDate
MATCH (res:RESULT)-[:COMPETED_IN]->(dis:DISCIPLINE)
MATCH (res)-[:PERFORM_DURING_ED]->(edi:EDITION)
MATCH (res)-[:PERFORMED_BY]->(ath:ATHLETE)
WHERE dis.name = "1.500 metres. Men" AND edi.year = "2018" AND res.result_date = LatestDate
RETURN DISTINCT ath.name AS Athlete, dis.name AS Discipline, res.result_date AS Date
ORDER BY Athlete ASC
                         """)
    for record in result:
        print(f"  {record['Discipline']}  {record['Date']}  {record['Athlete']}  ")          
driver.close() 


  1.500 metres. Men  2018-02-13  Aerchenghazi Xiakaini  
  1.500 metres. Men  2018-02-13  Alexis Contin  
  1.500 metres. Men  2018-02-13  Allan Dahl Johansson  
  1.500 metres. Men  2018-02-13  Andrea Giovannini  
  1.500 metres. Men  2018-02-13  Bart Swings  
  1.500 metres. Men  2018-02-13  Ben Donnelly  
  1.500 metres. Men  2018-02-13  Brian Hansen  
  1.500 metres. Men  2018-02-13  Denis Kuzin  
  1.500 metres. Men  2018-02-13  Denny Morrison  
  1.500 metres. Men  2018-02-13  Fyodor Mezentsev  
  1.500 metres. Men  2018-02-13  Haralds Silovs  
  1.500 metres. Men  2018-02-13  Jan Szymański  
  1.500 metres. Men  2018-02-13  Joey Mantia  
  1.500 metres. Men  2018-02-13  Ju Hyeong-Jun  
  1.500 metres. Men  2018-02-13  Kim Min-Seok  
  1.500 metres. Men  2018-02-13  Kjeld Nuis  
  1.500 metres. Men  2018-02-13  Koen Verweij  
  1.500 metres. Men  2018-02-13  Konrad Niedźwiedzki  
  1.500 metres. Men  2018-02-13  Konrád Nagy  
  1.500 metres. Men  2018-02-13  Livio Wenger  
  1.50

In [ ]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gbenitah"))   
with driver.session() as session:
    result = session.run("""
MATCH (res:RESULT)-[:COMPETED_IN]->(dis:DISCIPLINE)
MATCH (res)-[:PERFORM_DURING_ED]->(edi:EDITION)
MATCH (res)-[:PERFORMED_BY]->(ath:ATHLETE)
WHERE dis.name = $neodash_discipline_name AND edi.year = $neodash_edition_year 
WITH MAX(res.result_date) AS LatestDate
MATCH (res:RESULT)-[:COMPETED_IN]->(dis:DISCIPLINE)
MATCH (res)-[:PERFORM_DURING_ED]->(edi:EDITION)
MATCH (res)-[:PERFORMED_BY]->(ath:ATHLETE)
WHERE dis.name = "1.500 metres. Men" AND edi.year = "2018" AND res.result_date = LatestDate
RETURN DISTINCT ath.name AS Athlete, dis.name AS Discipline, res.result_date AS Date
ORDER BY Athlete ASC
                        """)
    for record in result:
        print(f"  {record['Discipline']}  {record['Date']}  {record['Athlete']}  ")          
driver.close() 